# Task 2 - Reusable Feature Pipeline
**Project:** Aura Workforce attrition | **Author:** Katarukonda Ashok | **Pipeline version:** `features-1.0.0`

**Goal:** turn the decisions from the Task 1 audit into ONE packaged pipeline that every later model reuses. It must fit once, save, load later and transform new rows in exactly the same way.

**Why:** if every model prepares data its own way, scores cannot be compared and the live service will not match training.

**Rule of the whole task:** the pipeline learns (medians, caps, categories, department pay) from the **training rows only**. Learning from the whole dataset before splitting is leakage.

Run top to bottom (`Run All`). The code lives in `src/features/`; this notebook explains and proves it.

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
# Find the project root (the folder that contains `src/`) no matter where the notebook is opened from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
DATA = ROOT / "data" / "raw" / "02_CortexHR_AI_ML_Dataset.xlsx"
TASK = ROOT / "Task2_Feature_Pipeline"
OUT = TASK / "outputs"; FIG = OUT / "figures"; TAB = OUT / "tables"
for d in (FIG, TAB): d.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

Project root: /home/claude/Aura_Phase1_Foundation


In [2]:
from src.features import FEATURE_VERSION, __version__
from src.features import config as cfg
from src.features.data import load_raw, load_clean_base, split_train_test
from src.features.pipeline import (build_feature_pipeline, fit_pipeline, transform_dataframe, output_columns,
                                   save_pipeline, load_pipeline)
from src.features.transformers import RowCleaner, SalaryTypoFixer, FeatureEngineer, Winsorizer
RESULTS = {}
print("Package version:", __version__, "| feature version:", FEATURE_VERSION)

Package version: 1.0.0 | feature version: features-1.0.0


## Step 1. Load, de-duplicate, split BEFORE fitting
Duplicates (decision D1) are removed first so one person cannot sit in both train and test. Then a **stratified** 80/20 split keeps the 14.6% leaver rate in both parts. The test part stays locked for Task 3.

In [3]:
raw = load_raw(DATA)
df = load_clean_base(DATA)
train, test = split_train_test(df)
print(f"raw rows {len(raw):,} -> after removing duplicates {len(df):,}")
print(f"train {len(train):,} rows (leavers {train[cfg.TARGET].mean():.2%})   |   test {len(test):,} rows (leavers {test[cfg.TARGET].mean():.2%})")
RESULTS.update(raw_rows=len(raw), rows_after_dedup=len(df), train_rows=len(train), test_rows=len(test),
               train_leaver_rate=float(train[cfg.TARGET].mean()), test_leaver_rate=float(test[cfg.TARGET].mean()))

raw rows 2,389 -> after removing duplicates 2,383
train 1,906 rows (leavers 14.64%)   |   test 477 rows (leavers 14.68%)


## Step 2. From audit decisions to code
Every decision in the Task 1 log has a home in the pipeline.

| Decision | Problem (Task 1) | Where it is handled |
|---|---|---|
| D1 | 6 duplicate rows | `data.drop_duplicates` (before the split) |
| D6 | 3 salaries 10x too high | `SalaryTypoFixer` (learns each role's usual pay band) |
| D7, D8, D9, D10, D11, D20 | negative tenure, attendance above 1, city spellings, wfh outside 0-1, negative experience, tenure above experience | `RowCleaner` (fixed rules, learns nothing) |
| D3 | skewed counts and pay | `Winsorizer` (caps at 1st/99th percentile learned on train) |
| M1-M4 | "not applicable" gaps | `FeatureEngineer` creates `never_promoted`, `no_raise_yet`, `not_reviewed_yet`, `engagement_missing` flags and fills the gaps sensibly |
| M5-M6 | small random gaps | median fill (`SimpleImputer`) and the category `Unknown` |
| D15, D16 | leakage columns, IDs | dropped by the final `ColumnTransformer` (`remainder="drop"`) |
| D17 | fit on train only | every step learns in `fit()`; the pipeline is fitted with `fit_pipeline(train)` |

**Engineered features asked for in the task:** attendance rate (`attendance_rate_3m`, plus `attendance_gap`), late count (`late_arrivals_3m`), average overtime (`avg_overtime_hours_per_month_3m`, `overtime_per_year_of_tenure`), tenure (`tenure_months`, `tenure_years`), salary versus department median (`pay_vs_dept_median`), months since last revision (`months_since_last_raise`).

## Step 3. The pipeline object

In [4]:
pipe = build_feature_pipeline()
for i, (name, step) in enumerate(pipe.steps, 1):
    print(f"{i}. {name:11s} -> {type(step).__name__}")
print()
enc = pipe.named_steps["encode"]
for name, trans, cols in enc.transformers:
    t = trans if isinstance(trans, str) else " -> ".join(type(s).__name__ for _, s in trans.steps)
    print(f"   encode/{name:5s}: {t:55s} {len(cols)} columns")

1. clean       -> RowCleaner
2. fix_salary  -> SalaryTypoFixer
3. engineer    -> FeatureEngineer
4. encode      -> ColumnTransformer

   encode/num  : Winsorizer -> SimpleImputer -> StandardScaler           29 columns
   encode/flag : passthrough                                             4 columns
   encode/cat  : SimpleImputer -> OneHotEncoder                          5 columns


## Step 4. Fit on the TRAINING rows only and look at what it learned

In [5]:
pipe = fit_pipeline(train)
fe, fixer = pipe.named_steps["engineer"], pipe.named_steps["fix_salary"]
print("Median pay per department (learned from train):")
display(pd.Series(fe.dept_median_).round(0).astype(int).rename("median pay (INR)").to_frame().T)
print("Role pay bands learned:", len(fixer.role_mid_), "roles | salary typos fixed in train:",
      (lambda c: (fixer.transform(c), fixer.n_fixed_last_)[1])(pipe.named_steps["clean"].transform(train)))
num = pipe.named_steps["encode"].named_transformers_["num"]
caps = pd.DataFrame({"low_cap": num.named_steps["cap"].low_, "high_cap": num.named_steps["cap"].high_}, index=cfg.NUMERIC_COLUMNS)
display(caps.loc[["tenure_months", "late_arrivals_3m", "avg_overtime_hours_per_month_3m", "pay_vs_dept_median"]].round(2))

Median pay per department (learned from train):


,D01,D02,D03,D04,D05,D06,D07,D08,D09,D10
median pay (INR),937566,965994,1023036,869676,775404,321480,501444,487848,486732,795552


Role pay bands learned: 48 roles | salary typos fixed in train: 2


,low_cap,high_cap
tenure_months,2.00,133.00
late_arrivals_3m,0.00,18.95
avg_overtime_hours_per_month_3m,0.00,24.69
pay_vs_dept_median,0.37,4.25


## Step 5. Transform: shape, missing values, column list

In [6]:
X_train = transform_dataframe(pipe, train)
X_test = transform_dataframe(pipe, test)
cols = output_columns(pipe)
groups = {"numeric (scaled)": [c for c in cols if c in cfg.NUMERIC_COLUMNS],
          "flags (0/1)": [c for c in cols if c in cfg.FLAG_COLUMNS],
          "categories (one-hot)": [c for c in cols if c not in cfg.NUMERIC_COLUMNS + cfg.FLAG_COLUMNS]}
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Missing values after transform: train", int(X_train.isna().sum().sum()), "| test", int(X_test.isna().sum().sum()))
display(pd.Series({k: len(v) for k, v in groups.items()}, name="columns").to_frame())
display(X_train.iloc[:3, :8].round(3))
leaks = [c for c in cols if any(c == b or c.startswith(b + "_") for b in cfg.LEAKAGE_COLUMNS + cfg.ID_COLUMNS + ["role_id", "observation_date"])]
print("Leakage / identifier columns in the output:", leaks or "none")
RESULTS.update(output_columns=len(cols), missing_after_transform=int(X_train.isna().sum().sum() + X_test.isna().sum().sum()),
               column_groups={k: len(v) for k, v in groups.items()}, leakage_columns_in_output=len(leaks))

Train: (1906, 74) | Test: (477, 74)
Missing values after transform: train 0 | test 0


,columns
numeric (scaled),29
flags (0/1),4
categories (one-hot),41


,job_level,tenure_months,total_experience_years,previous_employers_count,commute_distance_km,comp_ratio,raises_received,last_raise_pct
772,-0.441,-0.014,-0.892,1.546,-1.012,-0.113,0.445,-0.653
2256,-0.441,-0.691,-1.292,0.681,1.200,-0.273,-0.796,0.888
2160,0.441,-0.992,-0.247,-1.050,0.295,0.308,-0.796,1.357


Leakage / identifier columns in the output: none


## Step 6. See the clean-up working (before and after)
These are the exact problem rows found in Task 1.

In [7]:
cleaner = RowCleaner().fit(df)
before = df[df.employee_id.isin(["EMP-0248", "EMP-0280"])][["employee_id", "tenure_months", "total_experience_years", "work_location", "attendance_rate_3m"]]
after = cleaner.transform(df[df.employee_id.isin(["EMP-0248", "EMP-0280"])])[before.columns]
print("Negative tenure: BEFORE"); display(before)
print("AFTER RowCleaner"); display(after)

fx = SalaryTypoFixer().fit(df)
typos = df[df.employee_id.isin(["EMP-0034", "EMP-0433", "EMP-0601"])][["employee_id", "role_id", "annual_fixed_pay", "comp_ratio"]]
fixed = fx.transform(typos)
out = typos.assign(pay_after_fix=fixed.annual_fixed_pay)
print("Salary typos (fitted on the full table here only to demo; real use fits on train):"); display(out)

wf = df.copy(); wf["wfh_share_3m"].describe()
print("wfh_share_3m range  before:", round(df.wfh_share_3m.min(), 3), "to", round(df.wfh_share_3m.max(), 3),
      "| after:", cleaner.transform(df).wfh_share_3m.min(), "to", cleaner.transform(df).wfh_share_3m.max())
print("work_location values  before:", df.work_location.nunique(), "| after:", cleaner.transform(df).work_location.nunique())

Negative tenure: BEFORE


,employee_id,tenure_months,total_experience_years,work_location,attendance_rate_3m
108,EMP-0248,-4,8.0,Chennai,0.9531
125,EMP-0280,-5,11.7,Mumbai,0.8984


AFTER RowCleaner


,employee_id,tenure_months,total_experience_years,work_location,attendance_rate_3m
108,EMP-0248,0,8.0,Chennai,0.9531
125,EMP-0280,0,11.7,Mumbai,0.8984


Salary typos (fitted on the full table here only to demo; real use fits on train):


,employee_id,role_id,annual_fixed_pay,comp_ratio,pay_after_fix
14,EMP-0034,R015,4888200,0.966,488820.0
216,EMP-0433,R006,18671760,1.279,1867176.0
344,EMP-0601,R034,3606000,0.975,360600.0


wfh_share_3m range  before: -0.096 to 1.088 | after: 0.0 to 1.0
work_location values  before: 16 | after: 9


## Step 7. Why "fit on train only" matters (a concrete example)
`pay_vs_dept_median` divides pay by the department median. If the median were computed on ALL rows, the test rows would influence the numbers the model trains on. The two sets of medians are close but not equal, and that gap is exactly the leakage we avoid.

In [8]:
m_train = train.groupby("department_id").annual_fixed_pay.median()
m_all = df.groupby("department_id").annual_fixed_pay.median()
cmp = pd.DataFrame({"median_from_train": m_train, "median_from_all_rows": m_all})
cmp["difference"] = cmp.median_from_train - cmp.median_from_all_rows
display(cmp.round(0).astype(int).T)
print("Learned by the saved pipeline == train medians:", fe.dept_median_ == m_train.to_dict())

department_id,D01,D02,D03,D04,D05,D06,D07,D08,D09,D10
median_from_train,937566,965994,1023036,869676,775404,321480,501444,487848,486732,795552
median_from_all_rows,936294,958416,931740,873630,767934,326040,486732,530532,487068,737412
difference,1272,7578,91296,-3954,7470,-4560,14712,-42684,-336,58140


Learned by the saved pipeline == train medians: True


## Step 8. Save with a version, load later, get the same answer

In [9]:
art = OUT / "artifacts"
path = save_pipeline(pipe, train, art, extra={"n_test_rows_locked": len(test)})
print("Saved:", path.relative_to(ROOT), f"({path.stat().st_size/1024:.0f} KB)")
loaded, meta = load_pipeline(path)
same = transform_dataframe(pipe, test).equals(transform_dataframe(loaded, test))
print("Version in metadata:", meta["version"], "| rows it was trained on:", meta["n_training_rows"], "| output columns:", meta["n_output_columns"])
print("Loaded pipeline gives identical output to the original:", same)
print("Same input twice -> same output:", transform_dataframe(loaded, test).equals(transform_dataframe(loaded, test)))
display(pd.Series({k: meta[k] for k in ["version", "created_utc", "random_seed", "sklearn_version", "n_training_rows", "n_output_columns", "salary_typos_fixed_in_training"]}, name="metadata").to_frame())
RESULTS.update(version=meta["version"], loaded_matches_original=bool(same), salary_typos_fixed_in_training=meta["salary_typos_fixed_in_training"])

Saved: Task2_Feature_Pipeline/outputs/artifacts/features-1.0.0.joblib (13 KB)


Version in metadata: features-1.0.0 | rows it was trained on: 1906 | output columns: 74
Loaded pipeline gives identical output to the original: True


Same input twice -> same output: True


,metadata
version,features-1.0.0
created_utc,2026-10-14T04:10:14+00:00
random_seed,42
sklearn_version,1.8.0
n_training_rows,1906
n_output_columns,74
salary_typos_fixed_in_training,2


## Step 9. Run the tests

In [10]:
import subprocess
r = subprocess.run([sys.executable, "-m", "pytest", "tests/test_features.py", "-q", "--no-header", "--color=no"], cwd=ROOT, capture_output=True, text=True)
print(r.stdout[-1500:])
RESULTS["tests_summary"] = r.stdout.strip().splitlines()[-1]

...............                                                          [100%]
15 passed in 2.74s



## Step 10. One command from raw data to model-ready features

In [11]:
import subprocess
cmd = [sys.executable, "-m", "src.features.cli", "run", "--input", str(DATA), "--output-dir", str(OUT / "cli_demo")]
print("$ python -m src.features.cli run --input data/raw/02_CortexHR_AI_ML_Dataset.xlsx --output-dir artifacts")
r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
print(r.stdout)
print(sorted(p.name for p in (OUT / "cli_demo").iterdir()))
RESULTS["cli_ok"] = r.returncode == 0

$ python -m src.features.cli run --input data/raw/02_CortexHR_AI_ML_Dataset.xlsx --output-dir artifacts


{
  "version": "features-1.0.0",
  "rows_after_dedup": 2383,
  "train_rows": 1906,
  "test_rows": 477,
  "feature_columns": 74,
  "missing_values_after_transform": 0,
  "pipeline_file": "/home/claude/Aura_Phase1_Foundation/Task2_Feature_Pipeline/outputs/cli_demo/features-1.0.0.joblib"
}

['X_test_LOCKED.csv', 'X_train.csv', 'features-1.0.0.joblib', 'features-1.0.0.meta.json', 'run_summary.json', 'y_test_LOCKED.csv', 'y_train.csv']


## Summary
- The pipeline cleans, engineers, caps, fills, scales and encodes in one object (`features-1.0.0`).
- It is fitted on the training rows only, saved with `joblib`, and gives identical output after loading.
- Leakage columns and identifiers never reach the model; no missing values remain after the transform.
- Task 3 plugs this exact pipeline in front of each model and refits it inside every cross-validation fold.

In [12]:
(OUT / "results.json").write_text(json.dumps(RESULTS, indent=2, default=str))
import shutil; shutil.rmtree(OUT / "cli_demo", ignore_errors=True)   # demo files only; keep the folder tidy
print(json.dumps(RESULTS, indent=2, default=str)[:900])

{
  "raw_rows": 2389,
  "rows_after_dedup": 2383,
  "train_rows": 1906,
  "test_rows": 477,
  "train_leaver_rate": 0.14637985309548793,
  "test_leaver_rate": 0.14675052410901468,
  "output_columns": 74,
  "missing_after_transform": 0,
  "column_groups": {
    "numeric (scaled)": 29,
    "flags (0/1)": 4,
    "categories (one-hot)": 41
  },
  "leakage_columns_in_output": 0,
  "version": "features-1.0.0",
  "loaded_matches_original": true,
  "salary_typos_fixed_in_training": 2,
  "tests_summary": "15 passed in 2.74s",
  "cli_ok": true
}
